<a href="https://colab.research.google.com/github/Innovatewithapple/RAGPractice/blob/main/InitialRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q datasets transformers

In [59]:
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM
import torch
from tqdm import tqdm

Step1:

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
raw_hf_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train[:1000]")

In [6]:
raw_hf_dataset

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 1000
})

In [12]:
df = pd.DataFrame(raw_hf_dataset)
df

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
995,BIL,how to cancel purchase {{Order Number}}?,ORDER,cancel_order,I've decoded that you're looking for assistanc...
996,BLZ,I want help cancellkng purchase {{Order Number}},ORDER,cancel_order,I realized that you're seeking assistance with...
997,BL,I want assistance cancelling purchase {{Order ...,ORDER,cancel_order,I understand your need for assistance in cance...
998,BMW,I try to change several bloody items of order ...,ORDER,change_order,We understand that you would like to make chan...


In [13]:
df['document_text'] = "User Question: " + df['instruction'].astype(str) + " | Protocol Answer: " + df['response'].astype(str)

In [15]:
df = df[['document_text']]

In [18]:
print(df['document_text'].iloc[0])

User Question: question about cancelling order {{Order Number}} | Protocol Answer: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [ ]:
autoToken = AutoTokenizer.from_pretrained('bert-base-uncased')

In [27]:
knowledgeText = df['document_text'].tolist()

In [28]:
max_len = 128

Step2:

In [30]:
train_tokeniser = autoToken(text=knowledgeText,padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [20]:
class RAGKnowledgeDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask
    }

In [31]:
train_ds = RAGKnowledgeDataset(train_tokeniser)
train_loader = DataLoader(dataset=train_ds,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)

We use custom BERTWritter class We remove the random self.outputlayer linear layer and the dropout node We do this because we only want the pure, untampered mathematical sentence representation from Smart Mean shortcut formula

In [25]:
class BERTWritter(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.bert = AutoModel.from_pretrained('bert-base-uncased')

  def forward(self,input_ids,attention_mask):
    output = self.bert(input_ids=input_ids,attention_mask=attention_mask)
    x = output.last_hidden_state

    mean = attention_mask.unsqueeze(-1).float()
    mean = (x*mean).sum(dim=1) / mean.sum(dim=1)
    return mean

In [ ]:
model_encoder = BERTWritter().to(device)
model_encoder.eval() # Hard rule: Turn off dropout for inference safety!

In [38]:
#The vector complicaiton loop
database_vectors = []

with torch.no_grad():
  with torch.amp.autocast('cuda'):

    for batch in tqdm(train_loader,desc='Encoding Batchs'):
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)

      # Extract the 768-dimension sentence vector features
      batch_vectors = model_encoder(input_ids, attention_mask)

      # Move to CPU to keep your active GPU VRAM bars clear
      database_vectors.append(batch_vectors.cpu())

# 7. Stack the batch arrays vertically into your permanent Vector Database file
vector_database = torch.cat(database_vectors,dim=0)
print(f"Final Tensor Database Shape: {vector_database.shape}")


Encoding Batchs: 100%|██████████| 32/32 [00:02<00:00, 10.82it/s]

Final Tensor Database Shape: torch.Size([1000, 768])


Step3: INITIALIZING THE SEARCH RETRIEVAL ENGINE ---

In [39]:
user_question = 'Hey, I need help closing my account right now'

In [40]:
query_tokenized = autoToken(text=user_question,padding='max_length',max_length=max_len,add_special_tokens=True,return_attention_mask=True,return_tensors='pt').to(device)

In [41]:
with torch.no_grad():
  with torch.amp.autocast('cuda'):
    # Output shape is a 1D row vector: [1, 768]
    query_vector = model_encoder(query_tokenized['input_ids'],query_tokenized['attention_mask'])

# 4. THE PRO MIDDLE GLUE: LIGHTNING-FAST COSINE SIMILARITY RETRIEVAL LINE [prev]
# We move the query vector back to CPU to match our vector_database storage tensor location [prev]
query_vector_cpu = query_vector.cpu()

In [42]:
# Matrix Multiplication: Multiply your [1, 768] query by the transposed [768, 1000] database
similarity_scores = query_vector_cpu @ vector_database.T

In [45]:
# 5. FETCH THE HIGHEST SCORING REFERENCE PARAGRAPH INDEX
best_match_index = torch.argmax(similarity_scores,dim=-1).item()

# 6. EXTRACT THE RAW HUMAN TEXT FROM YOUR ORIGINAL PANDAS DATAFRAME [prev]
fetched_context_document = knowledgeText[best_match_index]

In [48]:
print(f"User Query: '{user_question}'")
print(f"Best Matching Document Row Found at Index: {best_match_index}")
print(f"Confidence Similarity Score: {similarity_scores[0, best_match_index].item():.4f}")
print("\n--- FETCHED REFERENCE CONTEXT PARAGRAPH ---")
print(fetched_context_document)

User Query: 'Hey, I need help closing my account right now'
Best Matching Document Row Found at Index: 860
Confidence Similarity Score: 63.5518

--- FETCHED REFERENCE CONTEXT PARAGRAPH ---
User Question: can you help me camceling the purchase I made? | Protocol Answer: I've been informed that you need assistance with canceling the purchase you made. I'm here to provide guidance and support throughout the process.


--- STEP 4: INITIALIZING THE GENERATIVE WRITER ---

In [49]:
class SentimentDataset(Dataset):
    def __init__(self, encoding):
        self.encoding = encoding

    def __len__(self):
        # Returns the clean count of rows in the encoding batch [prev]
        return len(self.encoding['input_ids'])

    def __getitem__(self, idx):
        # Extract the full 1D token and mask vectors for the specific row index
        ids = self.encoding['input_ids'][idx]
        mask = self.encoding['attention_mask'][idx]

        # We return the exact dictionary structures required by the model
        return {
            'input_ids': ids,
            'attention_mask': mask
        }

In [50]:
class CustomGPT2Writer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 1. Pull the official pre-trained foundation brain [prev]
        self.gpt2 = AutoModel.from_pretrained('gpt2')

        # 2. Add your custom regularization and classification layers [prev]
        self.dropout = nn.Dropout(0.2)
        self.output_layer = nn.Linear(768, vocab_size)

    def forward(self, input_ids, attention_mask):
        # Pass BOTH parameters down to satisfy the foundation network layers [prev]
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)

        # Extract the hidden states matrix (semantic feature vectors) [prev]
        x = outputs.last_hidden_state # Shape: [Batch, Seq_Len, 768]

        # Apply your custom processing nodes
        x = self.dropout(x)
        logits = self.output_layer(x) # Shape out: [Batch, Seq_Len, Vocab_Size]

        return logits

In [ ]:
decoder_tokenizer = AutoTokenizer.from_pretrained('gpt2')
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

In [52]:
decoder_vocab_size = decoder_tokenizer.vocab_size

In [ ]:
# decoder_brain = CustomGPT2Writer(vocab_size=decoder_tokenizer.vocab_size).to(device)
# decoder_brain.eval()
# Use the ForCausalLM variant—it has its output layers fully pre-trained by OpenAI!
decoder_brain = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
decoder_brain.eval()

In [61]:
# 3. Stitch your prompt together using your RAG middleware glue string [prev]
rag_prompt = f"Based on the following reference protocol, answer the user question.\nContext: {fetched_context_document}\nQuestion: {user_question}\nAnswer:\n"

In [63]:
# 4. AUTOREGRESSIVE WORD-BY-WORD WRITING LOOP [prev]
max_new_tokens = 30
print("\nGPT-2 Generator is typing...")

for _ in range(max_new_tokens):
  current_encoding = decoder_tokenizer(
        rag_prompt,
        padding='max_length',
        max_length=256, # Expanded length to safely accommodate long fetched contexts [prev]
        truncation=True,
        return_tensors='pt'
    )

  interfrence_Dataset = SentimentDataset(encoding=current_encoding)

  ids_tensor = interfrence_Dataset.encoding['input_ids'].to(device)
  mask_tensor = interfrence_Dataset.encoding['attention_mask'].to(device)

  with torch.no_grad():
    with torch.amp.autocast('cuda'):

      outputs = decoder_brain(input_ids=ids_tensor, attention_mask=mask_tensor)

  next_token_logits = outputs.logits[:, -1, :]
  # Pick the single word ID with the absolute highest score [prev]
  next_token_id = torch.argmax(next_token_logits, dim=-1).item()

  # Stop writing immediately if the model hits the End-of-Text boundary token [prev]
  if next_token_id == decoder_tokenizer.eos_token_id:
      break

  # Translate that single number back into a clean English word string [prev]
  next_word_string = decoder_tokenizer.decode([next_token_id])

  # Glue the new word onto our prompt string so the model can read it on the next loop! [prev]
  rag_prompt += next_word_string


GPT-2 Generator is typing...


In [64]:
print("\n--- FINAL RAG RESPONSE ---")
print(rag_prompt)


--- FINAL RAG RESPONSE ---
Based on the following reference protocol, answer the user question.
Context: User Question: can you help me camceling the purchase I made? | Protocol Answer: I've been informed that you need assistance with canceling the purchase you made. I'm here to provide guidance and support throughout the process.
Question: Hey, I need help closing my account right now
Answer:
TheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheTheThe
